In [3]:
import numpy as np
from scipy import signal
import math
import pandas as pd
from matplotlib import pyplot as plt
from fly2p_function_TQ.imaging_2p_preprocessing import combine_PB_corresponding_ROI
from tifffile import tifffile
pd_imaging_behavior_preprocessed = pd.read_csv('/home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/delta7_imaging_EPG_perturbation/delta7_syt6s_EPG_shi/all_file/TQfly239-001-imaging-normalized.csv')
F_array = np.array([pd_imaging_behavior_preprocessed[f'Raw_F_{i}'] for i in range(1, 17)]).T
label = tifffile.imread("/home/tianhaoqiu/Documents/GitHub/2p_analysis/napari_roi/delta7_imaging_EPG_perturbation/delta7_syt6s_EPG_shi/R55G08_syt6s_R60D05_shi/all/ROI_16_TQfly239-001.tif")
napari_roi = np.squeeze(label , axis=0) 
ROI_total =16
volume_cycle = len(pd_imaging_behavior_preprocessed)
volume_time = pd_imaging_behavior_preprocessed['Time_Stamp'][1]
volume_rate = 1/volume_time
time_array_imaging = np.arange(volume_cycle)/volume_rate
Z_score_array = np.zeros((F_array.shape[0], ROI_total))
F_sd = np.std(F_array, axis = 0)
F_mean = np.mean(F_array, axis = 0)
for z_index in range(ROI_total):
    Z_score_array [:,z_index] = (F_array[:,z_index] - F_mean[z_index])/F_sd[z_index]
for i in range (ROI_total):
    Z_score_array [:,i] = signal.medfilt(Z_score_array [:,i],kernel_size =3)
Z_score_array_8_roi = combine_PB_corresponding_ROI(dff_array_input = Z_score_array, napari_ROI = napari_roi , ROI_num = 8, mode = 2, time_array_imaging = time_array_imaging)
pd_imaging_behavior_preprocessed['Z_score_Roi_1'] = Z_score_array_8_roi[:,0]
pd_imaging_behavior_preprocessed['Z_score_Roi_2'] = Z_score_array_8_roi[:,1]
pd_imaging_behavior_preprocessed['Z_score_Roi_3'] = Z_score_array_8_roi[:,2]
pd_imaging_behavior_preprocessed['Z_score_Roi_4'] = Z_score_array_8_roi[:,3]
pd_imaging_behavior_preprocessed['Z_score_Roi_5'] = Z_score_array_8_roi[:,4]
pd_imaging_behavior_preprocessed['Z_score_Roi_6'] = Z_score_array_8_roi[:,5]
pd_imaging_behavior_preprocessed['Z_score_Roi_7'] = Z_score_array_8_roi[:,6]
pd_imaging_behavior_preprocessed['Z_score_Roi_8'] = Z_score_array_8_roi[:,7]

In [4]:
pd_imaging_behavior_preprocessed.to_csv('/home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/delta7_imaging_EPG_perturbation/delta7_syt6s_EPG_shi/updated/TQfly239-001.csv',encoding = 'utf-8', index=False)

In [ ]:
basename = os.path.basename(csv_file)

fly_full_id = basename.replace(".csv", "")
fly_short_id = fly_full_id.split("-imaging")[0]     # e.g. TQfly128-002

# Convert hyphen → underscore for ROI filenames
roi_id = fly_short_id.replace("-", "_")             # e.g. TQfly128_002

print(f"\n=== Processing {fly_full_id} ===")

roi_pattern = f"ROI_16_{roi_id}*.tif"               # ROI_16_TQfly128_002.tif
roi_matches = glob.glob(os.path.join(roi_dir, roi_pattern))

if len(roi_matches) == 0:
    print(f"⚠️  No ROI TIFF found for {roi_id}. Skipping.")
    continue

roi_file = roi_matches[0]
print(f"→ Using ROI file: {roi_file}")


In [11]:
import os
import glob
import numpy as np
from scipy import signal
import pandas as pd
from tifffile import imread
from fly2p_function_TQ.imaging_2p_preprocessing import combine_PB_corresponding_ROI


# -------------------------------
# USER-DEFINED DIRECTORIES
# -------------------------------
csv_input_dir = "/home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/dark"
roi_dir       = "/home/tianhaoqiu/Documents/GitHub/2p_analysis/napari_roi/dark_cl_imaging/GCaMP/EPG_syt7f_PB/all"
output_dir    = "/home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/updated/dark"

os.makedirs(output_dir, exist_ok=True)

# -------------------------------
# PROCESS EACH CSV
# -------------------------------
csv_files = sorted(glob.glob(os.path.join(csv_input_dir, "*.csv")))

for csv_file in csv_files:
    basename = os.path.basename(csv_file)

    # full ID (e.g. TQfly128-002-imaging-normalized)
    fly_full_id = basename.replace(".csv", "")

    # short ID used for ROI (e.g. TQfly128-002)
    fly_short_id = fly_full_id.split("-imaging")[0]

    print(f"\n=== Processing {fly_full_id} ===")

    # -------------------------------
    # Try BOTH hyphen and underscore ROI naming conventions
    # -------------------------------
    roi_hyphen_id = fly_short_id                    # e.g. TQfly128-002
    roi_underscore_id = fly_short_id.replace("-", "_")  # e.g. TQfly128_002

    roi_patterns = [
        f"ROI_16_{roi_hyphen_id}*.tif",
        f"ROI_16_{roi_underscore_id}*.tif"
    ]

    roi_matches = []
    for pattern in roi_patterns:
        matches = glob.glob(os.path.join(roi_dir, pattern))
        if matches:
            roi_matches = matches
            print(f"→ ROI match found using pattern: {pattern}")
            break

    if not roi_matches:
        print(f"⚠️  ERROR: No ROI TIFF found for either:")
        print(f"    ROI_16_{roi_hyphen_id}*.tif")
        print(f"    ROI_16_{roi_underscore_id}*.tif")
        print("    Skipping this file.\n")
        continue

    roi_file = roi_matches[0]
    print(f"→ Using ROI file: {roi_file}")

    # -------------------------------
    # LOAD CSV + RAW FLUORESCENCE
    # -------------------------------
    df = pd.read_csv(csv_file)
    F_array = np.array([df[f"Raw_F_{i}"] for i in range(1, 17)]).T

    # load ROI mask
    napari_roi = np.squeeze(imread(roi_file), axis=0)

    ROI_total = 16
    volume_cycle = len(df)
    volume_time = df["Time_Stamp"][1]
    volume_rate = 1 / volume_time
    time_array_imaging = np.arange(volume_cycle) / volume_rate

    # -------------------------------
    # COMPUTE Z-SCORES
    # -------------------------------
    Z = np.zeros((F_array.shape[0], ROI_total))
    F_sd = np.std(F_array, axis=0)
    F_mean = np.mean(F_array, axis=0)

    for i in range(ROI_total):
        Z[:, i] = (F_array[:, i] - F_mean[i]) / F_sd[i]
        Z[:, i] = signal.medfilt(Z[:, i], kernel_size=3)

    # -------------------------------
    # REDUCE TO 8 PB ROIs
    # -------------------------------
    Z8 = combine_PB_corresponding_ROI(
        dff_array_input=Z,
        napari_ROI=napari_roi,
        ROI_num=8,
        mode=2,
        time_array_imaging=time_array_imaging
    )

    # Add the 8 new Z-score columns
    for i in range(8):
        df[f"Z_score_Roi_{i+1}"] = Z8[:, i]

    # -------------------------------
    # SAVE UPDATED CSV WITH ORIGINAL NAME
    # -------------------------------
    save_path = os.path.join(output_dir, basename)
    df.to_csv(save_path, encoding="utf-8", index=False)

    print(f"✔ Saved updated CSV → {save_path}")

print("\n🎉 All files processed!")




=== Processing TQfly105-008-imaging-normalized ===
→ ROI match found using pattern: ROI_16_TQfly105-008*.tif
→ Using ROI file: /home/tianhaoqiu/Documents/GitHub/2p_analysis/napari_roi/dark_cl_imaging/GCaMP/EPG_syt7f_PB/all/ROI_16_TQfly105-008.tif
✔ Saved updated CSV → /home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/updated/dark/TQfly105-008-imaging-normalized.csv

=== Processing TQfly109-001-imaging-normalized ===
→ ROI match found using pattern: ROI_16_TQfly109-001*.tif
→ Using ROI file: /home/tianhaoqiu/Documents/GitHub/2p_analysis/napari_roi/dark_cl_imaging/GCaMP/EPG_syt7f_PB/all/ROI_16_TQfly109-001.tif
✔ Saved updated CSV → /home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/updated/dark/TQfly109-001-imaging-normalized.csv

=== Processing TQfly109-004-imaging-normalized ===
→ ROI match found using pattern: ROI_16_TQfly109-004*.tif
→ Using ROI file: /home/ti

✔ Saved updated CSV → /home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/updated/dark/TQfly116-006-imaging-normalized.csv

=== Processing TQfly118-001-imaging-normalized ===
→ ROI match found using pattern: ROI_16_TQfly118-001*.tif
→ Using ROI file: /home/tianhaoqiu/Documents/GitHub/2p_analysis/napari_roi/dark_cl_imaging/GCaMP/EPG_syt7f_PB/all/ROI_16_TQfly118-001.tif
✔ Saved updated CSV → /home/tianhaoqiu/Documents/GitHub/2p_analysis/preprocessing_output/normalized/dark_cl_Ca_imaging/EPG/EPG_PB(syt)/updated/dark/TQfly118-001-imaging-normalized.csv

🎉 All files processed!
